# Módulo 2 — Manipulação de Dados com Pandas

Este notebook foi desenvolvido como material de aula do **Módulo 2: Manipulação de Dados**, integrando conceitos teóricos, boas práticas e exemplos aplicados com dados reais de e-commerce brasileiro.

## Dataset utilizado: Olist Brazilian E-commerce

O dataset da **Olist** é um conjunto de dados públicos e reais, extraídos de uma das maiores plataformas de marketplace do Brasil. Ele contém informações sobre pedidos realizados entre 2016 e 2018, incluindo status de entrega, datas, avaliações dos clientes, produtos e pagamentos.

Neste módulo, vamos trabalhar principalmente com a tabela de pedidos (`olist_orders_dataset.csv`), que contém as seguintes colunas:

| Coluna | Descrição |
|---|---|
| `order_id` | Identificador único do pedido |
| `customer_id` | Identificador do cliente |
| `order_status` | Status do pedido (delivered, shipped, cancelled...) |
| `order_purchase_timestamp` | Data e hora da compra |
| `order_approved_at` | Data de aprovação do pagamento |
| `order_delivered_carrier_date` | Data de entrega à transportadora |
| `order_delivered_customer_date` | Data de entrega ao cliente |
| `order_estimated_delivery_date` | Data estimada de entrega |

## Pergunta norteadora deste módulo

> **"Como é o comportamento dos pedidos na Olist ao longo do tempo, e quais padrões conseguimos identificar antes mesmo de visualizar os dados?"**

Essa pergunta vai guiar todas as transformações que faremos aqui, preparando o terreno para as visualizações no Módulo 3 e a análise exploratória completa no Módulo 4.

## Ao final deste notebook, você será capaz de:

- Entender e usar **Series** e **DataFrames**
- Carregar dados de arquivos CSV
- Inspecionar a estrutura de um dataset real
- Identificar e tratar **valores ausentes**
- Remover **duplicatas** e corrigir **tipos de dados**
- Trabalhar com **datas e tempo** em Python
- **Filtrar**, **ordenar** e **transformar** dados
- **Agrupar** dados para responder perguntas de negócio
- Calcular **estatísticas descritivas** básicas

> 💡 **Dica:** Este é um módulo introdutório, mas usamos dados reais desde o início. Isso significa que vamos encontrar imperfeições típicas de bases do mundo real — e aprender a lidar com elas.

## Por que o Pandas é essencial em Ciência de Dados?

Em projetos reais, raramente recebemos os dados organizados e prontos para análise. É muito mais comum encontrar bases com:

- colunas com nomes pouco intuitivos
- datas armazenadas como texto
- valores ausentes em campos importantes
- registros duplicados
- tipos de dados incorretos
- informações espalhadas em múltiplas tabelas

O dataset da Olist é um exemplo claro disso. As datas de compra, aprovação e entrega vêm como texto, existem pedidos com status variados e há valores ausentes em colunas de data — o que faz sentido, já que um pedido cancelado jamais terá uma data de entrega.

O **Pandas** nos dá as ferramentas para transformar essa bagunça em informação organizada e pronta para análise. Nas próximas etapas, vamos fazer exatamente isso.

## 1. Importando as bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configurações de exibição: mostra todas as colunas e textos sem cortar
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 80)

print("Bibliotecas importadas com sucesso!")
print(f"Versão do Pandas: {pd.__version__}")

## 2. Estruturas fundamentais do Pandas

Antes de carregar o dataset, precisamos entender as duas estruturas de dados mais importantes do Pandas.

### 2.1 Series — uma dimensão

Uma **Series** é uma estrutura **unidimensional**: uma sequência de valores com um índice associado. Pense nela como uma única coluna de uma tabela.

No contexto da Olist, a lista de todos os status possíveis de um pedido pode ser representada como uma Series.

### 2.2 DataFrame — duas dimensões

Um **DataFrame** é uma estrutura **bidimensional**: linhas e colunas, exatamente como uma planilha. Cada coluna do DataFrame é, internamente, uma Series.

A tabela completa de pedidos da Olist, com todas as suas colunas de datas, status e identificadores, é um DataFrame.

In [ ]:
# Exemplo de Series: contagem de pedidos por status
# (valores fictícios para ilustrar o conceito antes de carregar o dataset real)
status_pedidos = pd.Series(
    [96478, 1107, 625, 301, 625, 148, 609, 8],
    index=[
        "delivered", "shipped", "cancelled",
        "unavailable", "invoiced", "processing",
        "created", "approved"
    ],
    name="quantidade_pedidos"
)

print("=== Series: status dos pedidos ===")
print(status_pedidos)
print()
print(f"Total de pedidos na amostra: {status_pedidos.sum():,}")
print(f"Status com mais pedidos: {status_pedidos.idxmax()}")
print(f"Percentual de entregues: {status_pedidos['delivered'] / status_pedidos.sum():.1%}")

Perceba que a Series nos permite calcular métricas rapidamente, como o percentual de pedidos entregues. Isso é útil quando queremos uma visão rápida de uma única variável.

In [ ]:
# Exemplo de DataFrame: amostra da estrutura da tabela de pedidos
# (dados fictícios para ilustrar o formato antes de usar o dataset real)
dados_exemplo = {
    "order_id": ["abc123", "def456", "ghi789", "jkl012", "abc123"],
    "customer_id": ["c001", "c002", "c003", "c004", "c001"],
    "order_status": ["delivered", "delivered", "cancelled", "shipped", "delivered"],
    "order_purchase_timestamp": [
        "2017-10-02 10:56:33", "2018-03-15 08:30:00",
        "2018-07-24 14:22:11", "2018-11-05 17:15:00",
        "2017-10-02 10:56:33"
    ],
    "order_delivered_customer_date": [
        "2017-10-10 18:30:00", "2018-03-22 12:00:00",
        np.nan, "2018-11-14 09:00:00", "2017-10-10 18:30:00"
    ],
    "order_estimated_delivery_date": [
        "2017-10-18 00:00:00", "2018-03-29 00:00:00",
        "2018-08-10 00:00:00", "2018-11-20 00:00:00",
        "2017-10-18 00:00:00"
    ]
}

df_exemplo = pd.DataFrame(dados_exemplo)

print("=== DataFrame: amostra da tabela de pedidos ===")
df_exemplo

Repare que já montamos esse exemplo com problemas intencionais típicos de bases reais:

- **Linha duplicada** (order_id `abc123` aparece duas vezes)
- **Valor ausente** (`NaN`) na data de entrega do pedido cancelado — faz sentido!
- **Datas como texto** — precisaremos converter para o tipo correto

Esses são exatamente os tipos de problemas que encontraremos no dataset real da Olist.

## 3. Carregando o dataset real

Agora vamos carregar o arquivo `olist_orders_dataset.csv`. Ele deve estar na pasta `datasets/` dentro da estrutura do repositório.

```
repositorio/
├── modulo_2/
│   └── notebook_modulo_2.ipynb  ← você está aqui
└── datasets/
    └── olist_orders_dataset.csv
```

Se precisar, ajuste o caminho na variável `CAMINHO_DATASET` abaixo.

In [ ]:
# Ajuste este caminho se necessário
CAMINHO_DATASET = "../datasets/olist_orders_dataset.csv"

# Lista de caminhos alternativos para facilitar diferentes organizações de pasta
caminhos_possiveis = [
    Path(CAMINHO_DATASET),
    Path("olist_orders_dataset.csv"),
    Path("datasets/olist_orders_dataset.csv"),
    Path("/content/olist_orders_dataset.csv"),  # Google Colab
]

arquivo = next((p for p in caminhos_possiveis if p.exists()), None)

if arquivo:
    df = pd.read_csv(arquivo)
    print(f"✅ Dataset carregado com sucesso!")
    print(f"   Arquivo: {arquivo}")
    print(f"   Linhas: {df.shape[0]:,}")
    print(f"   Colunas: {df.shape[1]}")
else:
    print("❌ Arquivo não encontrado.")
    print("   Verifique se o dataset está na pasta correta")
    print("   ou ajuste a variável CAMINHO_DATASET acima.")
    print()
    print("   Download: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce")

## 4. Inspeção inicial do dataset

Antes de qualquer análise, precisamos entender o que temos. Essa etapa de **inspeção inicial** é fundamental e deve ser sempre o primeiro passo ao trabalhar com um novo dataset.

As perguntas que queremos responder aqui são:
- Quantas linhas e colunas existem?
- Quais são os tipos de dados de cada coluna?
- Existem valores ausentes? Em quais colunas?
- Os primeiros registros fazem sentido?

In [ ]:
# Dimensões do dataset
linhas, colunas = df.shape
print(f"O dataset possui {linhas:,} pedidos e {colunas} colunas.")

In [ ]:
# Primeiros registros — uma olhada rápida no formato dos dados
df.head()

In [ ]:
# Tipos de dados e contagem de valores não-nulos por coluna
# Essa é uma das primeiras análises que fazemos em qualquer projeto
df.info()

### O que observamos no `.info()`?

Repare nas colunas de data — todas aparecem com tipo `object`, ou seja, **texto**. Isso é muito comum ao carregar CSVs: o Pandas não converte datas automaticamente. Vamos corrigir isso mais adiante.

Também é possível notar que algumas colunas têm **menos valores não-nulos** que o total de linhas. Isso indica a presença de valores ausentes — e no caso da Olist, isso tem uma explicação lógica: pedidos cancelados não terão data de entrega.

In [ ]:
# Contagem e percentual de valores ausentes por coluna
ausentes = df.isnull().sum()
percentual = (ausentes / len(df) * 100).round(2)

resumo_ausentes = pd.DataFrame({
    "valores_ausentes": ausentes,
    "percentual_%": percentual
}).sort_values("valores_ausentes", ascending=False)

print("=== Valores ausentes por coluna ===")
print(resumo_ausentes[resumo_ausentes["valores_ausentes"] > 0])

In [ ]:
# Últimos registros — útil para verificar se os dados têm consistência temporal
df.tail()

In [ ]:
# Verificando os valores únicos de uma coluna categórica
print("Status possíveis de um pedido:")
print(df["order_status"].unique())
print()
print("Contagem por status:")
print(df["order_status"].value_counts())

## 5. Limpeza de dados

Após inspecionar o dataset, passamos para a etapa de **limpeza**. Em projetos reais, essa etapa costuma ser a mais demorada — mas também é uma das mais importantes, pois garante que as análises posteriores sejam confiáveis.

Vamos trabalhar em três frentes:
1. Remoção de duplicatas
2. Tratamento de valores ausentes
3. Correção de tipos de dados (especialmente datas)

> ⚠️ **Boas práticas:** Nunca modifique o DataFrame original diretamente antes de entender os dados. Aqui, vamos trabalhar em uma cópia chamada `df_limpo`.

### 5.1 Duplicatas

In [ ]:
# Verificando duplicatas no DataFrame completo
duplicatas_totais = df.duplicated().sum()
print(f"Linhas completamente duplicadas: {duplicatas_totais}")

# Verificando duplicatas considerando apenas o identificador único
# Um order_id não deveria se repetir — cada pedido é único
duplicatas_order_id = df.duplicated(subset=["order_id"]).sum()
print(f"order_ids duplicados: {duplicatas_order_id}")

In [ ]:
# Criando uma cópia limpa e removendo duplicatas
df_limpo = df.drop_duplicates(subset=["order_id"]).copy()

print(f"Linhas antes da remoção: {len(df):,}")
print(f"Linhas após a remoção:   {len(df_limpo):,}")
print(f"Registros removidos:     {len(df) - len(df_limpo)}")

### 5.2 Valores ausentes — entendendo o contexto

Antes de tratar valores ausentes, é fundamental **entender por que eles existem**. Nem todo ausente é um erro de dados — às vezes, a ausência carrega uma informação.

No caso da Olist:
- `order_delivered_customer_date` ausente → o pedido **ainda não foi entregue** (pode estar em trânsito ou foi cancelado)
- `order_delivered_carrier_date` ausente → o pedido **não foi enviado** (cancelado, ou ainda em processamento)
- `order_approved_at` ausente → pagamento **não aprovado** ainda

Para os módulos seguintes, vamos manter os valores ausentes em colunas de data — eles fazem sentido e serão úteis para identificar pedidos com problemas.

In [ ]:
# Cruzando status com ausência na data de entrega para validar nossa hipótese
ausente_entrega = df_limpo[df_limpo["order_delivered_customer_date"].isnull()]

print("Status dos pedidos SEM data de entrega ao cliente:")
print(ausente_entrega["order_status"].value_counts())
print()
print("Confirmação: a grande maioria dos ausentes em datas de entrega são pedidos")
print("que realmente não foram entregues (cancelados, em trânsito etc).")

### 5.3 Conversão de tipos — datas como datetime

Esta é uma das transformações mais importantes neste dataset. As colunas de data estão armazenadas como **texto** (`object`), o que nos impede de fazer cálculos como:
- Quanto tempo durou a entrega?
- O pedido chegou antes ou depois da data estimada?
- Quantos pedidos foram feitos por mês?

Para responder essas perguntas, precisamos converter as colunas para o tipo **`datetime`** do Pandas.

In [ ]:
# Colunas de data no dataset
colunas_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convertendo para datetime com tratamento de erros
# errors='coerce' converte valores inválidos para NaT (Not a Time) em vez de gerar erro
for coluna in colunas_data:
    df_limpo[coluna] = pd.to_datetime(df_limpo[coluna], errors="coerce")

# Verificando os tipos após a conversão
print("Tipos das colunas de data após a conversão:")
print(df_limpo[colunas_data].dtypes)

In [ ]:
# Exemplo do que mudou: comparando o formato antes e depois
print("Exemplo de valor na coluna 'order_purchase_timestamp':")
print(df_limpo["order_purchase_timestamp"].iloc[0])
print(f"Tipo: {type(df_limpo['order_purchase_timestamp'].iloc[0])}")

## 6. Transformação de dados — criando novas colunas

Com as datas convertidas corretamente, podemos agora criar **indicadores derivados** — colunas calculadas a partir das existentes — que são fundamentais para qualquer análise de e-commerce.

Vamos criar:
- **`prazo_entrega_dias`**: quantos dias o cliente esperou pelo pedido
- **`atraso_dias`**: quantos dias de atraso em relação à data estimada (negativo = adiantado)
- **`entregou_no_prazo`**: indicador booleano de cumprimento do prazo
- **`mes_compra`** e **`ano_compra`**: para análise temporal
- **`dia_semana_compra`**: para identificar padrões de comportamento

In [ ]:
# Prazo de entrega: diferença entre data de entrega e data de compra
# .dt.days extrai apenas o número de dias da diferença
df_limpo["prazo_entrega_dias"] = (
    df_limpo["order_delivered_customer_date"] - df_limpo["order_purchase_timestamp"]
).dt.days

# Atraso: diferença entre data real de entrega e data estimada
# Valores positivos = atrasado | Negativos = adiantado
df_limpo["atraso_dias"] = (
    df_limpo["order_delivered_customer_date"] - df_limpo["order_estimated_delivery_date"]
).dt.days

# Flag: entregou no prazo ou não?
# Só faz sentido para pedidos que foram entregues
df_limpo["entregou_no_prazo"] = df_limpo["atraso_dias"] <= 0

# Extraindo componentes temporais da data de compra
df_limpo["ano_compra"] = df_limpo["order_purchase_timestamp"].dt.year
df_limpo["mes_compra"] = df_limpo["order_purchase_timestamp"].dt.month
df_limpo["dia_semana_compra"] = df_limpo["order_purchase_timestamp"].dt.day_name()

# Visualizando o resultado
colunas_novas = [
    "order_id", "order_status",
    "prazo_entrega_dias", "atraso_dias",
    "entregou_no_prazo", "ano_compra", "mes_compra", "dia_semana_compra"
]
df_limpo[colunas_novas].head(10)

In [ ]:
# Verificando se a criação das colunas faz sentido
# Pedidos entregues: devem ter prazo_entrega_dias preenchido
entregues = df_limpo[df_limpo["order_status"] == "delivered"]

print(f"Total de pedidos entregues: {len(entregues):,}")
print(f"Desses, com prazo calculado: {entregues['prazo_entrega_dias'].notna().sum():,}")
print()
print(f"Percentual que entregou no prazo: {entregues['entregou_no_prazo'].mean():.1%}")

## 7. Filtragem e seleção de dados

Filtrar dados é uma das operações mais frequentes em análise. Com o Pandas, podemos selecionar subconjuntos do DataFrame usando condições lógicas.

In [ ]:
# Filtrando apenas pedidos com status 'delivered'
df_entregues = df_limpo[df_limpo["order_status"] == "delivered"].copy()
print(f"Pedidos entregues: {len(df_entregues):,}")

In [ ]:
# Filtrando pedidos com atraso (atraso_dias > 0)
df_atrasados = df_entregues[df_entregues["atraso_dias"] > 0]
print(f"Pedidos entregues com atraso: {len(df_atrasados):,}")
print(f"Percentual de atrasos: {len(df_atrasados) / len(df_entregues):.1%}")

In [ ]:
# Combinando condições com & (E) e | (OU)
# Pedidos do ano de 2018 com prazo de entrega acima de 20 dias
filtro_complexo = (
    (df_entregues["ano_compra"] == 2018) &
    (df_entregues["prazo_entrega_dias"] > 20)
)

resultado = df_entregues[filtro_complexo]
print(f"Pedidos de 2018 com entrega acima de 20 dias: {len(resultado):,}")

In [ ]:
# Usando .isin() para filtrar múltiplos valores de uma vez
# Pedidos que não foram concluídos com sucesso
status_problematicos = ["cancelled", "unavailable", "processing"]
df_problemas = df_limpo[df_limpo["order_status"].isin(status_problematicos)]

print(f"Pedidos com status problemático: {len(df_problemas):,}")
print()
print(df_problemas["order_status"].value_counts())

## 8. Ordenação de dados

In [ ]:
# Os 10 pedidos com maior tempo de entrega
top_demorados = (
    df_entregues
    .sort_values("prazo_entrega_dias", ascending=False)
    [["order_id", "order_purchase_timestamp", "order_delivered_customer_date", "prazo_entrega_dias"]]
    .head(10)
)

print("Top 10 pedidos com maior prazo de entrega:")
top_demorados

In [ ]:
# Os 10 pedidos entregues com maior antecedência (mais adiantados)
top_adiantados = (
    df_entregues
    .sort_values("atraso_dias", ascending=True)
    [["order_id", "atraso_dias", "prazo_entrega_dias"]]
    .head(10)
)

print("Top 10 pedidos entregues com mais antecedência (atraso negativo = adiantado):")
top_adiantados

## 9. Agrupamentos e agregações

O `groupby()` é um dos recursos mais poderosos do Pandas. Ele permite **agrupar** os dados por uma ou mais variáveis e **agregar** os resultados com funções como `mean()`, `sum()`, `count()` etc.

Em e-commerce, isso é essencial para responder perguntas como:
- Qual mês concentrou mais pedidos?
- Em qual dia da semana os clientes mais compram?
- Como variou o prazo médio de entrega ao longo do tempo?

In [ ]:
# Volume de pedidos por ano e mês
pedidos_por_mes = (
    df_limpo
    .groupby(["ano_compra", "mes_compra"])
    .agg(total_pedidos=("order_id", "count"))
    .reset_index()
    .sort_values(["ano_compra", "mes_compra"])
)

print("Volume de pedidos por ano e mês:")
pedidos_por_mes

In [ ]:
# Prazo médio de entrega e taxa de pontualidade por mês de compra
desempenho_mensal = (
    df_entregues
    .groupby(["ano_compra", "mes_compra"])
    .agg(
        total_pedidos=("order_id", "count"),
        prazo_medio_dias=("prazo_entrega_dias", "mean"),
        taxa_pontualidade=("entregou_no_prazo", "mean")
    )
    .round(2)
    .reset_index()
)

desempenho_mensal["taxa_pontualidade"] = (
    desempenho_mensal["taxa_pontualidade"] * 100
).round(1).astype(str) + "%"

print("Desempenho de entrega por mês:")
desempenho_mensal

In [ ]:
# Pedidos por dia da semana
# Qual dia da semana concentra mais compras?
ordem_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
nomes_pt = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]
mapa_dias = dict(zip(ordem_dias, nomes_pt))

pedidos_por_dia = (
    df_limpo
    .groupby("dia_semana_compra")
    .agg(total=("order_id", "count"))
    .reindex(ordem_dias)  # garante a ordem correta da semana
    .rename(index=mapa_dias)
)

print("Pedidos por dia da semana:")
pedidos_por_dia

## 10. Estatística descritiva

Com os dados transformados, podemos calcular medidas estatísticas básicas que nos ajudam a entender a distribuição de variáveis numéricas.

Vamos focar no `prazo_entrega_dias` — uma das métricas mais importantes de qualidade de serviço no e-commerce.

In [ ]:
# Resumo estatístico completo do prazo de entrega
# (apenas pedidos entregues, para ter dados válidos)
prazo = df_entregues["prazo_entrega_dias"].dropna()

print("=== Estatísticas do prazo de entrega (em dias) ===")
print(f"  Contagem:      {prazo.count():,} pedidos")
print(f"  Média:         {prazo.mean():.1f} dias")
print(f"  Mediana:       {prazo.median():.1f} dias")
print(f"  Desvio padrão: {prazo.std():.1f} dias")
print(f"  Mínimo:        {prazo.min():.0f} dias")
print(f"  Máximo:        {prazo.max():.0f} dias")
print(f"  Q1 (25%):      {prazo.quantile(0.25):.1f} dias")
print(f"  Q3 (75%):      {prazo.quantile(0.75):.1f} dias")

In [ ]:
# O método .describe() resume tudo isso em uma linha
print("Resumo via .describe():")
df_entregues[["prazo_entrega_dias", "atraso_dias"]].describe().round(2)

### Interpretando os resultados

- A **média** é maior que a **mediana** → a distribuição é assimétrica à direita, com alguns pedidos levando muito mais tempo que a maioria
- O **desvio padrão** alto indica grande variação no prazo de entrega
- O valor **máximo** muito elevado sugere possíveis outliers ou casos excepcionais

Essa diferença entre média e mediana é um sinal clássico de **dados com outliers** — algo que exploraremos graficamente no Módulo 3.

In [ ]:
# Categorização do prazo de entrega em faixas
# Transformando variável contínua em categorias interpretáveis
df_entregues = df_entregues.copy()
df_entregues["faixa_prazo"] = pd.cut(
    df_entregues["prazo_entrega_dias"],
    bins=[0, 7, 14, 21, 30, float("inf")],
    labels=["Até 1 semana", "1-2 semanas", "2-3 semanas", "3-4 semanas", "Acima de 30 dias"]
)

print("Distribuição por faixa de prazo:")
print(df_entregues["faixa_prazo"].value_counts().sort_index())

## 11. Salvando o DataFrame tratado

Após todo o processo de limpeza e transformação, é boa prática salvar o DataFrame tratado para uso nos módulos seguintes. Isso evita reprocessar os dados toda vez.

In [ ]:
# Salvando o DataFrame limpo e enriquecido
caminho_saida = Path("../datasets/olist_orders_tratado.csv")
caminho_saida.parent.mkdir(parents=True, exist_ok=True)

df_limpo.to_csv(caminho_saida, index=False)
print(f"✅ Dataset tratado salvo em: {caminho_saida}")
print(f"   Linhas: {len(df_limpo):,}")
print(f"   Colunas: {df_limpo.shape[1]}")
print()
print("Colunas disponíveis no arquivo tratado:")
for col in df_limpo.columns:
    print(f"  - {col} ({df_limpo[col].dtype})")

## 12. Perguntas analíticas para os próximos módulos

Com os dados limpos e enriquecidos, estamos prontos para avançar. Aqui estão perguntas que serão exploradas visualmente no Módulo 3 e analiticamente no Módulo 4:

1. **Como evoluiu o volume de pedidos mês a mês?** *(tendência de crescimento da Olist)*
2. **Qual é a distribuição dos prazos de entrega?** *(existem outliers? a maioria entrega em menos de 2 semanas?)*
3. **Em qual dia da semana os clientes mais compram?** *(padrão de comportamento de consumo)*
4. **A taxa de pontualidade melhorou ou piorou ao longo do tempo?** *(evolução da qualidade operacional)*
5. **Pedidos feitos em períodos de Black Friday ou Natal têm atrasos maiores?** *(impacto de sazonalidade)*

Essas perguntas serão a base do nosso projeto de **Análise Exploratória de Dados** no Módulo 4.

## Lista de Exercícios — Módulo 2

Pratique os conceitos aprendidos respondendo às perguntas abaixo. Use o DataFrame `df_limpo` ou `df_entregues` conforme necessário.

---

**1.** Quantos pedidos únicos existem no dataset? Use o método adequado para verificar.

**2.** Qual é o status de pedido mais comum? E o menos comum?

**3.** Crie uma nova coluna chamada `tempo_aprovacao_horas` que calcule, em horas, o tempo entre a compra e a aprovação do pagamento.

**4.** Filtre apenas os pedidos realizados em **novembro de 2017** (possível Black Friday). Quantos foram?

**5.** Qual foi o mês com maior volume de pedidos? Em qual ano?

**6.** Calcule a **mediana** do prazo de entrega separadamente para pedidos entregues **no prazo** e pedidos **atrasados**. O que você observa?

**7.** Agrupe por `mes_compra` e calcule o total de pedidos e o prazo médio de entrega. Existe algum padrão sazonal?

**8.** Filtre os pedidos com prazo de entrega **acima de 60 dias**. Quantos são? O que há de incomum nesses registros?

**9.** Usando `.value_counts(normalize=True)`, calcule o percentual de pedidos por `faixa_prazo`.

**10.** Identifique o mês com **pior taxa de pontualidade** (menor percentual de entregas no prazo). Em qual período isso aconteceu e o que pode ter causado?


## Conclusão

Neste módulo, percorremos o fluxo completo de manipulação de dados com Pandas usando um dataset real de e-commerce brasileiro:

- carregamos os dados e fizemos uma **inspeção cuidadosa**
- identificamos e removemos **duplicatas**
- entendemos o **contexto dos valores ausentes** antes de tratá-los
- convertemos **datas de texto para datetime**, desbloqueando cálculos temporais
- criamos **indicadores derivados** como prazo de entrega e flag de atraso
- usamos **filtros**, **ordenação** e **agrupamentos** para extrair insights iniciais
- calculamos **estatísticas descritivas** e identificamos a assimetria nos dados

A grande lição deste módulo: **a qualidade da análise depende diretamente da qualidade da preparação dos dados**. Um dado mal tipado ou não tratado pode levar a conclusões completamente erradas.

No **Módulo 3**, vamos transformar tudo isso em visualizações — histogramas, boxplots, séries temporais e heatmaps — que revelarão padrões impossíveis de perceber apenas olhando para números.